# 00 - Colab setup: ADSP 32018 medallion-RAG

Run this at the top of any Colab session (training, EDA, experiments). It:
1. mounts Google Drive,
2. copies the frozen data snapshot to Colab's local disk and verifies its checksum,
3. installs the project code from the team repo (optional),
4. sets up a run folder on Drive so outputs survive the session.

**Access:** you must be a member of the **NLP_Project** Shared Drive (ask Ryan). It appears in Colab under `/content/drive/Shareddrives/NLP_Project`.

**Rule:** never open `project.duckdb` directly from `/content/drive` (sync can lock or corrupt it - docs/DECISIONS.md D3). This notebook copies it to `/content` first.

## 1. Settings

In [ ]:
TEAM_DIR = '/content/drive/Shareddrives/NLP_Project'
SNAPSHOT_VERSION = 'v1'
REPO_URL = ''          # team GitHub repo; leave empty to skip installing the code
PROBLEM = 'p1'         # p1 | p2 | p3 - decides where outputs are saved
RUN_NAME = 'setup_check'  # short name for this run, e.g. 'distilbert_lr2e-5'
WORK_DIR = '/content/work'

## 2. Mount Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 3. Copy, verify and unzip the data snapshot
Fails loudly if the checksum does not match the `.sha256` published with the snapshot.

In [ ]:
import hashlib
import shutil
import zipfile
from pathlib import Path

snap_dir = Path(TEAM_DIR) / '01_data_snapshots' / SNAPSHOT_VERSION
zip_src = next(snap_dir.glob('data_snapshot_*.zip'))
expected = zip_src.with_suffix('.sha256').read_text().split()[0]

work = Path(WORK_DIR)
work.mkdir(parents=True, exist_ok=True)
zip_local = work / zip_src.name
if not zip_local.exists():
    shutil.copy(zip_src, zip_local)

digest = hashlib.sha256()
with zip_local.open('rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        digest.update(chunk)
assert digest.hexdigest() == expected, 'Checksum mismatch - re-copy the snapshot'

if not (work / 'data' / 'project.duckdb').exists():
    with zipfile.ZipFile(zip_local) as zf:
        zf.extractall(work)
print(f'{zip_src.name}: checksum OK, unzipped to {work / "data"}')

## 4. Install the project code (optional)
Installs the `medrag` package from the team repo so `from medrag import db` works.

In [ ]:
import os
import subprocess

os.environ['MEDRAG_DATA_DIR'] = str(work / 'data')
if REPO_URL:
    repo = work / 'repo'
    if not repo.exists():
        subprocess.run(['git', 'clone', '-q', REPO_URL, str(repo)], check=True)
    subprocess.run(['pip', 'install', '-q', '-e', str(repo)], check=True)
    print('medrag installed from', REPO_URL)
else:
    print('REPO_URL empty - skipping code install')

## 5. Quick check: open the database read-only

In [ ]:
import duckdb

con = duckdb.connect(os.environ['MEDRAG_DATA_DIR'] + '/project.duckdb', read_only=True)
display(con.execute(
    'SELECT table_schema, COUNT(*) AS n_tables FROM information_schema.tables '
    'GROUP BY 1 ORDER BY 1'
).df())

## 6. Run folder on Drive
Save checkpoints, logs and metrics under `RUN_DIR` so they survive the Colab session. Training outputs go to `05_models/`; evaluation outputs go to `04_results/`. Always write `config.json` so every number is traceable.

In [ ]:
import json
from datetime import date

RUN_DIR = Path(TEAM_DIR) / '05_models' / PROBLEM / f'{date.today()}_{RUN_NAME}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
config = {'problem': PROBLEM, 'run_name': RUN_NAME, 'snapshot': zip_src.name,
          'snapshot_sha256': expected, 'repo_url': REPO_URL}
(RUN_DIR / 'config.json').write_text(json.dumps(config, indent=2))
print('Save outputs to:', RUN_DIR)